# Windowed Analysis Tutorial

This tutorial provides a deep dive into Neurodent's windowed analysis capabilities for extracting features from continuous EEG data.

## Overview

Windowed Analysis Results (WAR) is the core feature extraction system in Neurodent. It:

1. Divides continuous EEG data into time windows
2. Computes features for each window
3. Aggregates results across time and channels
4. Provides filtering and quality control methods

This approach is efficient for long recordings and enables parallel processing.

In [ ]:
import logging
import tempfile
from datetime import datetime
from pathlib import Path

from IPython.display import display

from neurodent import (
    AnimalOrganizer,
    AnimalAnalyzer,
    WindowAnalysisResult,
    ZeitgeberAnalysisResult,
    constants,
    set_channel_map,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

In [ ]:
set_channel_map({
    "LMot": ["C-015", "D-015"],
    "RMot": ["C-016", "D-016"],
    "LBar": ["C-014", "D-014"],
    "RBar": ["C-017", "D-017"],
    "LHip": ["C-012", "D-012"],
    "RHip": ["C-019", "D-019"],
    "LAud": ["C-009", "D-009"],
    "RAud": ["C-022", "D-022"],
    "LVis": ["C-010", "D-010"],
    "RVis": ["C-021", "D-021"],
})

# Per-animal genotype and sex. Every WindowAnalysisResult re-reads this at construction.
constants.ANIMAL_METADATA = {
    "A10": {"genotype": "WT", "sex": "Male"},
    "F22": {"genotype": "KO", "sex": "Female"},
}

## 1. Feature Categories

Neurodent extracts four main categories of features:

### Linear Features (per channel)
Single-value metrics for each channel in each time window:

In [ ]:
# Available linear features
print("Linear features:")
for feature in constants.LINEAR_FEATURES:
    print(f"  - {feature}")

# Examples:
# - rms: Root mean square amplitude
# - logrms: Log of RMS amplitude
# - ampvar: Amplitude variance
# - psdtotal: Total power spectral density
# - psdslope: Slope of PSD on log-log scale

### Band Features (per frequency band)
Features computed for each frequency band (delta, theta, alpha, beta, gamma):

In [ ]:
# Available band features
print("\nBand features:")
for feature in constants.BAND_FEATURES:
    print(f"  - {feature}")

# Frequency bands
print("\nFrequency bands:")
for band, (lo, hi) in constants.FREQ_BANDS.items():
    print(f"  {band.capitalize()}: {lo}-{hi} Hz")

### Matrix Features (connectivity)
Features measuring relationships between channels:

In [ ]:
# Available matrix features
print("\nMatrix features:")
for feature in constants.MATRIX_FEATURES:
    print(f"  - {feature}")

# Examples:
# - cohere: Spectral coherence between channel pairs
# - pcorr: Pearson correlation between channels

## 2. Computing Windowed Analysis

### Basic Usage

In [ ]:
# Using the included example data.
# The paired ColMajor .bin + Meta .csv recordings are a couple of minutes long, which is
# enough for windowing to be meaningful. (The .edf files in the same folder are only 5
# seconds long and would yield a single window.)
animal_id = "A10"

ao = AnimalOrganizer(
    pattern=[
        "../../.tests/integration/data/{animal}/*_ColMajor.bin",
        "../../.tests/integration/data/{animal}/*_Meta.csv",
    ],
    animal_id=animal_id,
    lro_kwargs={
        "mode": "si",  # mode options: 'si' (SpikeInterface), 'mne', or None
        "extract_func": "../../tests/integration/readers.py:read_bin_csv_pair",
        "manual_datetimes": datetime(2023, 12, 13, 12, 0),
    },
)

analyzer = AnimalAnalyzer(ao)
analyzer.compute_bad_channels()

# Compute all features
war_all = analyzer.compute_windowed_analysis(
    features=['all'],
    exclude=['nspike', 'lognspike'],  # Exclude spike features if no spikes
    multiprocess_mode='serial'
)

print(f"Computed {len(war_all.result)} windows")

### Access information in WindowedAnalysis object

You can access the summary of WindowedAnalysis object using the `get_info` method, and access the computed features in the WindowedAnalysis object using the `get_result` method.

In [ ]:
war_info = war_all.get_info()
print(war_info)

result = war_all.get_result(
    features=["all"],
    exclude=['nspike', 'lognspike']
)
display(result)

### Selective Feature Computation

For faster processing, compute only needed features:

In [ ]:
# Compute specific features
war_selective = AnimalAnalyzer(ao).compute_windowed_analysis(
    features=['rms', 'logrms', 'psdband', 'cohere'],
    multiprocess_mode='serial'
)

result = war_selective.get_result(
    features=['rms', 'logrms', 'psdband', 'cohere']
)

print(f"Computed features: {list(result.keys())}")

### Parallel Processing

`multiprocess_mode` accepts two values: `'serial'` and `'dask'`. Serial is the right choice
for debugging and for small recordings like the example data; Dask distributes fragment
computation across a cluster and is what the Snakemake pipeline uses for real datasets.

In [ ]:
# Dask mode, for distributed computing. Neurodent starts a local cluster if one is not
# already running, so this also works on a single machine.
war_dask = AnimalAnalyzer(ao).compute_windowed_analysis(
    features=['rms', 'psdband'],
    multiprocess_mode='dask'
)

print(f"Dask run produced {len(war_dask.result)} windows")

## 3. Data Quality and Filtering

### Method Chaining (Recommended)

Apply multiple filters in sequence:

In [ ]:
war_filtered = (
    war_all
    .filter_logrms_range(z_range=3)           # Remove outliers (±3 SD)
    .filter_high_rms(max_rms=500)             # Remove high amplitude artifacts
    .filter_low_rms(min_rms=10)               # Remove low amplitude periods
    # .filter_high_beta(max_beta_prop=0.4)      # Remove high beta activity
    .filter_reject_channels_by_session()      # Reject bad channels
)

print("Filtering completed!")

### Configuration-Driven Filtering

Alternative approach using configuration dictionary:

In [ ]:
filter_config = {
    'logrms_range': {'z_range': 3},
    'high_rms': {'max_rms': 500},
    'low_rms': {'min_rms': 10},
    # 'high_beta': {'max_beta_prop': 0.4},
    'reject_channels_by_session': {},
    'morphological_smoothing': {'smoothing_seconds': 4.0}
}

war_filtered_config = war_all.apply_filters(
    filter_config,
    min_valid_channels=3
)

### Available Filters

Chainable methods on a `WindowAnalysisResult`:

- `filter_logrms_range(z_range)`: Remove outliers based on log RMS
- `filter_high_rms(max_rms)`: Remove high amplitude artifacts
- `filter_low_rms(min_rms)`: Remove low amplitude periods
- `filter_high_beta(max_beta_prop)`: Remove high beta activity (muscle artifacts)
- `filter_reject_channels_by_session()`: Identify and reject bad channels

Morphological smoothing has no standalone method — it is a post-processing step available
through the `morphological_smoothing` key of `apply_filters()`, as shown above.

## 4. Data Aggregation

Average across time windows, producing a single row per group (e.g., per recording session
and light/dark phase). Channel information is preserved.

`aggregate_time_windows()` modifies the object in place and returns `None`, so work on a
copy if you still need the per-window data afterwards — as the circadian section below does.

In [ ]:
# Aggregate time windows on a copy, keeping war_filtered per-window for later sections
war_aggregated = war_filtered.copy()
war_aggregated.aggregate_time_windows()

print(f"Before aggregation: {len(war_filtered.result)} rows")
print(f"After aggregation:  {len(war_aggregated.result)} rows")

## 5. Channel Management

### Reorder and Pad Channels

Ensure consistent channel ordering across animals:

In [ ]:
# Define standard channel order
standard_channels = [
    "LMot", "RMot",  # Motor cortex
    "LBar", "RBar",  # Barrel cortex
    "LAud", "RAud",  # Auditory cortex
    "LVis", "RVis",  # Visual cortex
    "LHip", "RHip"   # Hippocampus
]

war_filtered.reorder_and_pad_channels(
    standard_channels,
    use_abbrevs=True  # Use abbreviated channel names
)

print(f"Channels: {war_filtered.channel_names}")

## 6. Accessing Computed Features

WAR objects store features in a pandas DataFrame. Use `get_result()` to retrieve features with full channel information, or `get_channel_averaged_result()` to average across channels (or channel pairs for connectivity features), producing scalar values per time window.

In [ ]:
# Get the full result DataFrame
result_df = war_filtered.get_result(
    features=['rms', 'psdband', 'cohere']
)
print(f"Result columns: {list(result_df.columns)}")
print(f"Result shape: {result_df.shape}")
print(f"\nRMS values (per-channel arrays):\n{result_df['rms'].iloc[0]}")

# Get channel-averaged result (scalars per time window)
df_avg = war_filtered.get_channel_averaged_result(
    features=['logrms', 'logpsdband', 'zcohere']
)
print(f"\nChannel-averaged columns: {list(df_avg.columns)}")
print(f"\nSample logrms value: {df_avg['logrms'].iloc[0]}")  # single float

## 7. Metadata and Grouping Variables

WAR objects contain metadata for grouping and analysis:

In [ ]:
# Access metadata
print(f"Animal ID: {war_filtered.animal_id}")
print(f"Genotype: {war_filtered.genotype}")
print(f"Animal days: {war_filtered.animaldays}")
print(f"Channel names: {war_filtered.channel_names}")

## 8. Circadian Analysis (ZeitgeberAnalysisResult)

Once your data is filtered and metadata (like genotype and timestamps) is verified, you can analyze circadian rhythms. 

The `ZeitgeberAnalysisResult` wrapper uses this metadata to:
1.  **Shift Timestamps**: Converts absolute timestamps to Zeitgeber Time (ZT), where ZT0 is "Lights On". The ZT coordinate lands in the `zt_minutes` column.
2.  **Define Baseline**: Subtracts a baseline period (e.g., the first 12 hours of the light phase) to normalize the data, adding `*_nobase` columns.

Baseline subtraction only applies to numeric columns, so use `get_channel_averaged_result()`
rather than `get_result()` when you want the `*_nobase` columns — `get_result()` keeps one
array of per-channel values per cell.

Double-plotted 48-hour actograms are produced at render time by `ZeitgeberPlotter`, not by
this wrapper. For plotting these results, see the **[Visualization Tutorial](visualization.ipynb)**.

In [ ]:
# Wrap the result for circadian analysis
zar = ZeitgeberAnalysisResult(
    war_filtered,
    baseline_hours=12,
    zeitgeber_shift_hours=6,
)

# Channel-average so baseline subtraction has numeric columns to work on
df_zar = zar.get_channel_averaged_result(features=['rms'])

timestamps = war_filtered.result['timestamp']
print(f"Original Time Range: {timestamps.min()} to {timestamps.max()}")
print(f"ZT Coordinate Range: {df_zar.zt_minutes.min()} to {df_zar.zt_minutes.max()} min")
print(f"Baseline-Corrected Columns: {[c for c in df_zar.columns if '_nobase' in c]}")

## 9. Saving and Loading

Save WAR objects for later analysis:

In [ ]:
# Use a temporary directory for demonstration purposes
with tempfile.TemporaryDirectory() as tmpdir:
    output_path = Path(tmpdir) / animal_id
    output_path.mkdir(parents=True, exist_ok=True)

    # Save WAR as parquet + json
    war_filtered.save_parquet_and_json(output_path)
    print(f"Saved to {output_path}")

    # Load WAR
    war_loaded = WindowAnalysisResult.load_parquet_and_json(output_path)
    print(f"Loaded {len(war_loaded.result)} rows from {output_path}")

## 10. Best Practices

### Feature Selection
- Start with basic features (rms, psdband) before computing expensive ones (cohere, psd)
- Exclude spike features if you don't have spike data
- Use selective feature computation for faster iteration

### Filtering
- Always inspect data before and after filtering
- Use conservative thresholds initially, then adjust
- Consider biological significance (e.g., high beta may indicate muscle artifacts)

### Processing
- Use serial mode for debugging and small recordings
- Use Dask for larger datasets and cluster computing

### Quality Control
- Check channel consistency across animals
- Verify metadata (genotype, day, etc.)
- Save intermediate results frequently

## Summary

This tutorial covered:

1. Feature categories and types
2. Computing windowed analysis with different options
3. Data quality control and filtering
4. Channel management and standardization
5. Accessing computed features
6. Metadata and grouping variables
7. Saving and loading results
8. Best practices

## Next Steps

- **[Visualization Tutorial](visualization.ipynb)**: Plot and analyze WAR results
- **[Spike Analysis Tutorial](spike_analysis.ipynb)**: Integrate spike-sorted data